# <span style="color:orange"> **NOTES** </span>
## <span style="color:yellow"> ***WEEK 1*** </span>
### <span style="color:red"> **Neural Networks - Building Micrograd** </span>
### <span style="color:green"> What is micrograd? </span>
It is an automatic gradient (autograd) engine that uses backpropagation



In [167]:
import math
import numpy as np 
import matplotlib.pyplot as plt
import random

%matplotlib inline 
#this command displays plots inside notebook instead of creating new window

### <span style="color:green"> How is the gradient calculated? </span>
By arithmetic approximation using the definition of derivative


### <span style="color:green"> The Value class </span>

In [168]:
class Value:
    def __init__(self,data,_children=(),_op='',label=''):
        self.data=data
        self._prev=set(_children)
        self._op=_op
        self.label=label
        self.grad=0 #at initialization we assume that the value object doesnt affect the loss function so gradiennt is 0 by default
        self._backward=lambda:None #by default this _backward function should do nothing at  the nodes
        #basically the task of backward is to store how to backflow the derivative from the self node to the chidren nodes

    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self,other):
        other=other if isinstance(other,Value) else Value(other)
        sum=Value(self.data+other.data,(self,other),'+')

        def _backward():
            #we add the derivatives as otherwise if we assign the derivatives then the derivative contributions from various possible branhes get overwritten instead of getting added
            self.grad+=1.0*sum.grad
            other.grad+=1.0*sum.grad
            
        sum._backward=_backward
        return sum 
    
    def __radd__(self,other):
        return self+other
    
    def __neg__(self):#-self
        return self*-1
    
    def __sub__(self,other):#self-other
        return self+(-other) 

    def __rsub__(self,other):
        return -1*(self-other)       
    
    def __mul__(self,other):
        other=other if isinstance(other,Value) else Value(other)
        product=Value(self.data*other.data,(self,other),'*')

        def _backward():
            self.grad+=product.grad*other.data
            other.grad+=product.grad*self.data

        product._backward=_backward
        return product
    
    def __rmul__(self,other): #when Python fails to execute other*self
        return self*other
    
    def __truediv__(self, other):
        return self*other**-1
    
    def __pow__(self,other):
        assert isinstance(other,(int,float)),"Supports float/int powers for now"
        out= Value(self.data**other,(self,),f'**{other}')

        def _backward():
            self.grad+=other*(self.data**(other-1))*out.grad
        out._backward=_backward
        return out
        
    def tanh(self):
        x=self.data
        t=(math.exp(2*x)-1)/(math.exp(2*x)+1)
        out=Value(t,(self,),'tanh')
        
        def _backward():
            self.grad+=(1-t**2)*out.grad
        out._backward=_backward
        return out
    
    def exp(self):
        x=self.data
        t=math.exp(x)
        out=Value(t,(self,),'exp')

        def _backward():
            self.grad+=out.grad*t
        out._backward=_backward
        return out
    
    def backward(self):
        #this function calls the _backward starting from a node down the topological sort of the subtree rooted at that node
        self.grad=1.0

        #topological sorting of the DAG
        topo=[]
        visited=set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad=1.0
        for node in reversed(topo):
            node._backward()


#### <span style="color:green">Closures in Python</span>
For nested functions in Python like the_backward in the Value class, the function creates a closure, i.e. stores the references used inside it from the surrounding environment. When the function is called it follows LEGB for accessing variables. Here the E(Enclosing)  is the scope corresponding to closure.

#### <span style="color:green">Visualising the process</span>

In [169]:
from graphviz import Digraph

In [170]:
#the following function initialises the nodes and edges sets of a graph
def trace(root):
    nodes,edges=set(),set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child,v))
                build(child)
    build(root)
    return nodes, edges

#the following function visualises the graph
def draw_dot(root):
    dot=Digraph(format='svg', graph_attr={'rankdir':'LR'}) #LR=left to right
    nodes,edges=trace(root)
    for n in nodes:
        uid=str(id(n))
        # for any node in the graph create a rectangular ('record') node for it
        dot.node(name=uid,label="{%s | data %.4f | grad %.4f}" % (n.label,n.data,n.grad),shape='record')
        if n._op:
            # if this value is the result of some operation, create an op node for it
            dot.node(name = uid + n._op, label=n._op)
            #and connect this node to it
            dot.edge(uid+n._op, uid)
    
    for n1,n2 in edges:
        #connect n1 to the op node of n2
        dot.edge(str(id(n1)),str(id(n2))+n2._op)

    return dot

### <span style="color:green"> Backpropagation </span>
1. The gradient contributions to the gradient of a node from different branches get added up.
2. A plus node basically distributes the gradient among all the children nodes
3. A product node multiplies the parent grad by the other child and stores this sum as the grad of the child
 

### <span style="color:green"> Neural Network (Multi Layer Perceptron [MLP]) </span>

In [171]:
class Neuron:

    def __init__(self,nin):
        self.w=[Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b=Value(random.uniform(-1,1))

    def __call__(self,x):
        # w*x+b
        act=sum((wi*xi for wi,xi in zip(self.w,x)),self.b) #the zip basically forms list of tuples
        out=act.tanh() #squicification function
        return out
    
    def parameters(self):
        return self.w+[self.b]
    
class Layer:
    
    def __init__(self,nin,nout):
        self.neurons=[Neuron(nin) for _ in range(nout)]

    def __call__(self,x):
        outs=[n(x) for n in self.neurons]
        return outs[0] if len(outs)==1 else outs
    
    def parameters(self):
        return [x for neuron in self.neurons for x in neuron.parameters() ]
    
class MLP:

    def __init__(self,nin,nouts):
        self.layers=[]
        for nout in nouts:
            self.layers.append(Layer(nin,nout))
            nin=nout
        
    def __call__(self,x):
        for layer in self.layers:
            x=layer(x)
        return x
    
    def parameters(self):
        return [x for layer in self.layers for x in layer.parameters()]

#### <span style="color:green"> Example of Training Loop </span>


In [172]:
xs=[[1.0,2.0,3.0],
    [1.0,4.0,2.0],
    [7.0,13.0,8.0],
    [17.0,18.0,19.0],
    [5.0,7.0,0.0]]
ys=[1.0,-1.0,-1.0,1.0,-1.0]
mlp=MLP(3,[4,4,1])

#### <span style="color:green">Training Loop</span>

In [173]:
N=1000
for k in range(N):
    lr=0.1*(1-1.0*k/N)
    for p in mlp.parameters():
        p.grad=0.0
    ypreds=[mlp(x) for x in xs]
    #forward pass
    loss=sum((yp-y)**2 for y,yp in zip(ys,ypreds))
    #backward pass
    loss.backward()
    #learning
    for p in mlp.parameters():
        p.data-=lr*p.grad
loss
    



Value(data=0.000454530301550426)